# 05 · Loss functions: measuring how wrong a prediction is

*Neural Networks from Scratch: Build, Train and Understand AI Models*, notebook 5 of 12

Your network can already turn inputs into predictions, but nothing yet tells it whether those predictions are any good. A **loss function** turns "how wrong" into a single number that training can push down. In this notebook you write the four loss functions that the rest of the course relies on, and you see exactly how each one punishes a bad prediction.

## What you will build

- `mse(pred, target)`: the mean squared error, for predicting numbers;
- `binary_cross_entropy(p, y)`: the loss for yes-or-no predictions;
- `cross_entropy(probs, y)`: the loss for choosing one of several classes;
- `cross_entropy_from_logits(Z, y)`: the same loss computed safely from raw scores with the log-sum-exp trick.

## Learning objectives

By the end of this notebook you will be able to:

1. explain what a loss function is and why training needs one number per batch;
2. compute mean squared error, binary cross-entropy and multiclass cross-entropy by hand and with NumPy;
3. explain why a confident wrong prediction costs far more than an uncertain one;
4. recognise when `np.exp` overflows or `np.log` receives zero, and fix both with log-sum-exp;
5. choose a suitable loss for regression, binary classification and multiclass classification.

### How to work through this notebook

Run the cells from top to bottom. When you reach a **✍️ Your turn** cell, type the function yourself and then run the check cell beneath it:

- ✅ means your function matches the reference, and from then on the notebook runs **your** code;
- ✍️ means you have not written it yet, and ❌ means it does not match yet. In both cases the notebook quietly uses a hidden reference version, so you can always keep going and come back later.

Each reference solution sits in a collapsed cell just above the function you type. Open it only when you are stuck.

## Dependencies

- **Tier 1:** NumPy for every calculation and Matplotlib for the plots. Nothing else is imported, so the notebook runs in JupyterLite in your browser, on the course JupyterHub server and in Google Colab.
- Nothing is downloaded: the examples use small arrays written in the cells and spiral data generated by `make_spirals`.
- Every random number comes from a seeded generator, so your numbers match the ones quoted in the text.

## Setup

Run this cell first. It imports NumPy and Matplotlib, creates a seeded random generator and defines `practice`, the helper that checks each function you write against a reference.

In [ ]:
# ── Setup ─ run this cell first ────────────────────────────────────────────
import numpy as np

rng = np.random.default_rng(0)


def _close(got, want, atol):
    if isinstance(want, dict):
        return isinstance(got, dict) and got.keys() == want.keys() and all(
            _close(got[key], want[key], atol) for key in want
        )
    if isinstance(want, (list, tuple)):
        return isinstance(got, (list, tuple)) and len(got) == len(want) and all(
            _close(g, w, atol) for g, w in zip(got, want)
        )
    try:
        got_array, want_array = np.asarray(got, dtype=float), np.asarray(want, dtype=float)
    except (TypeError, ValueError):
        return got == want
    return got_array.shape == want_array.shape and np.allclose(got_array, want_array, atol=atol)


def _short(value):
    if isinstance(value, tuple):
        return "(" + ", ".join(_short(item) for item in value) + ")"
    if isinstance(value, np.ndarray):
        text = np.array2string(value, precision=4, threshold=8, separator=", ").replace("\n", "")
    else:
        text = repr(value)
    return text if len(text) < 120 else text[:117] + "..."


def practice(name, yours, reference, cases, atol=1e-6):
    """Check your function against the reference on a few inputs.

    Returns the function the rest of the notebook should use: yours once it
    matches, otherwise the reference -- so you can always keep going.
    """
    try:
        for args in cases:
            got, want = yours(*args), reference(*args)
            if not _close(got, want, atol):
                print(f"❌ {name}: for input {_short(args)}")
                print(f"   you returned {_short(got)}")
                print(f"   expected     {_short(want)}")
                print(f"   The rest of the notebook uses the reference {name} until yours matches.")
                return reference
    except NotImplementedError:
        print(f"✍️  {name}: not written yet. Type it in the cell above and run this cell again.")
        print("   Until then the rest of the notebook uses the reference version.")
        return reference
    except Exception as error:  # a typo in the learner's code must not stop the lesson
        print(f"❌ {name} raised {type(error).__name__}: {error}")
        print(f"   The rest of the notebook uses the reference {name} until yours runs.")
        return reference
    print(f"✅ {name}: matches the reference on {len(cases)} test case{'s' if len(cases) != 1 else ''}. From here on the notebook runs YOUR {name}.")
    return yours

import matplotlib.pyplot as plt

## 📦 Recap: what you built earlier

The hidden cell below re-defines the pieces from notebooks 01–04 under their usual names, so this notebook works on its own: `relu`, `relu_grad`, `sigmoid`, `softmax`, `dense_forward`, `init_network`, `forward`, `predict`, `accuracy` and `make_spirals`. Run it; open it only if you want to look back at the code.

In [ ]:
# 📦 Recap: the pieces from notebooks 01–04, under their usual names.

def relu(Z):
    """ReLU, element by element: max(0, Z)."""
    return np.maximum(0.0, Z)


def relu_grad(Z):
    """The slope of ReLU: 1.0 where Z > 0, else 0.0 (we take 0 at Z = 0)."""
    return (Z > 0).astype(float)


def sigmoid(Z):
    """Numerically stable logistic function 1 / (1 + exp(-Z)), element by element."""
    return np.exp(-np.logaddexp(0.0, -Z))


def softmax(Z):
    """Row-wise softmax of logits Z (batch, classes). Subtracting each row's maximum avoids overflow."""
    exps = np.exp(Z - Z.max(axis=1, keepdims=True))
    return exps / exps.sum(axis=1, keepdims=True)


def dense_forward(X, W, b):
    """Z = X @ W + b. X: (batch, n_in), W: (n_in, n_out), b: (n_out,) -> Z: (batch, n_out)."""
    return X @ W + b


def init_network(sizes, rng):
    """He-initialised layers for sizes such as [2, 64, 3]: a list of {"W": (n_in, n_out), "b": (n_out,)}."""
    return [
        {"W": rng.normal(0.0, np.sqrt(2.0 / n_in), (n_in, n_out)), "b": np.zeros(n_out)}
        for n_in, n_out in zip(sizes[:-1], sizes[1:])
    ]


def forward(network, X):
    """ReLU hidden layers and a softmax output.

    Returns (probs, cache): probs has shape (batch, classes), and cache[i] = (A_prev, Z) for layer i.
    """
    cache = []
    A = X
    for index, layer in enumerate(network):
        Z = dense_forward(A, layer["W"], layer["b"])
        cache.append((A, Z))
        A = softmax(Z) if index == len(network) - 1 else relu(Z)
    return A, cache


def predict(network, X):
    """The predicted class index for every row of X: shape (batch,)."""
    probs, _ = forward(network, X)
    return probs.argmax(axis=1)


def accuracy(network, X, y):
    """The fraction of rows of X whose predicted class equals the label in y."""
    return float(np.mean(predict(network, X) == y))


def make_spirals(n_per_class, n_classes, noise, rng):
    """n_classes interleaved spiral arms in 2-D, sorted by class.

    Returns X with shape (n_per_class * n_classes, 2) and integer labels y with
    shape (n_per_class * n_classes,). noise is the spread of the random angle jitter.
    """
    radius = np.linspace(0.05, 1.0, n_per_class)
    X_parts, y_parts = [], []
    for k in range(n_classes):
        angle = 2 * np.pi * k / n_classes + 6.0 * radius + rng.normal(0.0, noise, n_per_class)
        X_parts.append(np.column_stack([radius * np.cos(angle), radius * np.sin(angle)]))
        y_parts.append(np.full(n_per_class, k))
    return np.concatenate(X_parts), np.concatenate(y_parts)

## 1 · What a loss function does

A network produces predictions. A **loss function** compares them with the correct answers and returns one number: zero for perfect predictions and larger for worse ones. Training then adjusts the weights to make that number smaller, so the loss is what defines "better" for your network. Choose the wrong loss and the network will dutifully get better at the wrong thing.

For a batch of $n$ examples we compute a loss $\ell_i$ for each example and report the **mean**,

$$L = \frac{1}{n} \sum_{i=1}^{n} \ell_i ,$$

so that the loss of a batch of 32 examples can be compared with the loss of a batch of 1,000. Every function in this notebook returns that mean as a plain Python `float`.

## 2 · Mean squared error, for predicting numbers

When the target is a number, such as a price or a temperature, the natural error is the difference $\hat{y}_i - y_i$ between the prediction and the target. Squaring it makes every error positive and punishes large errors much more than small ones:

$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (\hat{y}_i - y_i)^2 .$$

**Shapes.** `pred` and `target` have the same shape: `(batch,)` for one output per example, or `(batch, outputs)` for several. The mean runs over **every** entry, and the result is a single number.

The cell below shows the ingredients for three predictions.

In [ ]:
pred = np.array([2.5, 0.0, 2.0])
target = np.array([3.0, -0.5, 2.0])
errors = pred - target
print("errors:        ", errors)
print("squared errors:", errors ** 2)

### ✍️ Your turn: `mse`

Write `mse(pred, target)`:

- **inputs:** `pred` and `target`, NumPy arrays of the same shape;
- **output:** the mean of the squared differences over every entry, as a Python `float`.

Hints: `(pred - target) ** 2` squares element by element, `np.mean` averages every entry of an array whatever its shape, and `float(...)` turns a NumPy number into a plain Python number.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def mse_reference(pred, target):
    return float(np.mean((pred - target) ** 2))

In [ ]:
def mse(pred, target):
    """Mean squared error over every entry. pred and target: arrays of the same shape. Returns a float."""
    # ✍️ Your code here (one line)
    raise NotImplementedError

In [ ]:
mse = practice("mse", mse, mse_reference, cases=[
    (np.array([2.5, 0.0, 2.0]), np.array([3.0, -0.5, 2.0])),
    (np.array([[1.0, 2.0], [3.0, 4.0]]), np.array([[1.5, 2.0], [2.0, 6.0]])),
    (np.zeros(4), np.zeros(4)),  # edge case: perfect predictions cost nothing
])

**Expected behaviour.** For the three predictions above the errors are $-0.5$, $0.5$ and $0$, so the MSE is $(0.25 + 0.25 + 0)/3 \approx 0.1667$. Perfect predictions give exactly $0$. Notice that doubling every error multiplies the MSE by four: squaring is what makes large mistakes dominate the average.

## 3 · Binary cross-entropy, for yes-or-no predictions

For a yes-or-no question ("is this email spam?") a network outputs one probability $p_i = \sigma(z_i)$ that the answer is *yes*, and the label is $y_i \in \{0, 1\}$. The probability the model gave to what actually happened is $p_i$ when $y_i = 1$ and $1 - p_i$ when $y_i = 0$. A good model gives the truth a high probability, so we charge the negative logarithm of that probability, which is small when the probability is close to 1 and huge when it is close to 0:

$$\text{BCE} = -\frac{1}{n} \sum_{i=1}^{n} \Big[ y_i \log p_i + (1 - y_i) \log(1 - p_i) \Big].$$

For each example exactly one of the two terms is switched on by the label. The sum is the negative log-likelihood of the labels under the model's probabilities, so minimising it is the same as maximum-likelihood estimation.

**Clipping.** If a probability is exactly $0$ or $1$, `np.log(0)` returns `-inf` and the loss becomes infinite, or `nan` because $0 \times (-\infty)$ is undefined. We therefore clip the probabilities into $[10^{-12},\ 1 - 10^{-12}]$ before taking logarithms. The clip changes nothing for sensible probabilities and caps the loss of a single example at $-\log 10^{-12} \approx 27.63$.

**Shapes.** `p` and `y` both have shape `(batch,)`, and `y` holds 0s and 1s.

In [ ]:
p = np.array([0.9, 0.2, 0.6])   # predicted probability of "yes"
y = np.array([1, 0, 0])         # what actually happened
prob_of_truth = np.where(y == 1, p, 1 - p)
print("probability given to the truth:", prob_of_truth)
print("minus its logarithm:           ", np.round(-np.log(prob_of_truth), 4))

### ✍️ Your turn: `binary_cross_entropy`

Write `binary_cross_entropy(p, y)`:

- **inputs:** `p`, predicted probabilities of shape `(batch,)`, and `y`, labels of shape `(batch,)` that are 0 or 1;
- **output:** the mean binary cross-entropy as a Python `float`, computed after clipping `p` into $[10^{-12},\ 1 - 10^{-12}]$.

Hints: start with `p = np.clip(p, 1e-12, 1 - 1e-12)`, then follow the formula with `np.log` and `np.mean`. Do not forget the minus sign.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below.

In [ ]:
def binary_cross_entropy_reference(p, y):
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))

In [ ]:
def binary_cross_entropy(p, y):
    """Mean binary cross-entropy. p: (batch,) probabilities of "yes", y: (batch,) 0/1 labels. Returns a float."""
    # ✍️ Your code here (two lines: clip, then the formula)
    raise NotImplementedError

In [ ]:
binary_cross_entropy = practice("binary_cross_entropy", binary_cross_entropy, binary_cross_entropy_reference, cases=[
    (np.array([0.9, 0.2, 0.6]), np.array([1, 0, 0])),
    (np.array([0.5, 0.5]), np.array([1, 0])),
    (np.array([1.0, 0.0]), np.array([0, 1])),  # edge case: certain and wrong, so the clip matters
])

**Expected behaviour.** The three predictions above cost $0.1054$, $0.2231$ and $0.9163$, so the batch loss is about $0.4149$. A coin-flip prediction, $p = 0.5$, costs $\log 2 \approx 0.6931$ whatever the label. A prediction that is certain and wrong costs $27.63$ thanks to the clip, instead of `inf`.

## 4 · Confident and wrong versus uncertain

Cross-entropy only looks at $q$, the probability that the model gave to the **true** answer, and charges $-\log q$. That curve drives all of classification training, so it is worth seeing in numbers. The table compares it with the squared error $(1 - q)^2$ for a range of predictions about one example whose true label is 1.

In [ ]:
prob_of_truth = np.array([0.99, 0.9, 0.7, 0.5, 0.3, 0.1, 0.01, 0.0001])
print(f"{'q = p(truth)':>13} {'cross-entropy':>14} {'squared error':>14}")
for q in prob_of_truth:
    ce = binary_cross_entropy(np.array([q]), np.array([1]))
    se = mse(np.array([q]), np.array([1.0]))
    print(f"{q:>13g} {ce:>14.4f} {se:>14.4f}")

In [ ]:
q = np.linspace(0.001, 1.0, 400)
fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.plot(q, -np.log(q), color="#2a78d6", linewidth=2, label=r"cross-entropy $-\log q$")
ax.plot(q, (1 - q) ** 2, color="#eb6834", linewidth=2, label=r"squared error $(1 - q)^2$")
ax.set_xlabel("q, the probability given to the true class")
ax.set_ylabel("loss for one example")
ax.set_ylim(0, 7)
ax.grid(alpha=0.3)
ax.legend()
ax.set_title("Confident mistakes are expensive under cross-entropy")
plt.show()

**What to notice.**

- An uncertain prediction ($q = 0.5$) costs $0.69$, and a confident correct one ($q = 0.99$) costs almost nothing ($0.01$).
- A confident **wrong** prediction is punished hard, and without limit: $q = 0.01$ costs $4.61$ and $q = 0.0001$ costs $9.21$. Every tenfold drop in the probability of the truth adds another $\log 10 \approx 2.30$.
- The squared error can never exceed $1$ here, so it barely distinguishes "quite wrong" from "certain and wrong". The steep cross-entropy curve keeps pushing a confidently wrong model to change its mind, which is why classifiers are trained with it.

### 🧪 Try it: one confident mistake in a batch of ten

Nine predictions give the truth a probability of $0.95$; one gives it only `q_mistake`. Change `q_mistake` and run the cell again to see how much of the batch loss comes from that single example.

In [ ]:
q_mistake = 0.001        # try 0.3, 1e-6 and 0.0, then run the cell again
q_batch = np.array([0.95] * 9 + [q_mistake])
y_batch = np.ones(10)
per_example = -np.log(np.clip(q_batch, 1e-12, 1 - 1e-12))
print(f"batch loss: {binary_cross_entropy(q_batch, y_batch):.3f}")
print(f"share of the loss caused by the single mistake: {per_example[-1] / per_example.sum():.0%}")

With `q_mistake = 0.001` the batch loss is $0.737$, and the one mistake causes $94\%$ of it. At `0.0` the clip caps that example at $27.63$; without the clip the whole batch loss would be `inf`, and training would break.

## 5 · Cross-entropy for several classes

With $K$ classes the softmax layer outputs one row of probabilities per example: `probs` has shape `(batch, K)` and each row sums to 1. The labels `y` are **integer class indices** of shape `(batch,)`. The loss is again minus the logarithm of the probability given to the true class, averaged over the batch:

$$\text{CE} = -\frac{1}{n} \sum_{i=1}^{n} \log P_{i,\,y_i} .$$

If you write the labels as one-hot rows $Y$ (`np.eye(K)[y]`), the same loss is $-\frac{1}{n}\sum_i \sum_k Y_{ik} \log P_{ik}$: the one-hot row switches on exactly one term per example. Picking that entry directly is simpler and faster. `probs[np.arange(n), y]` takes column $y_i$ from row $i$, for every $i$ at once.

With $K = 2$ classes this is exactly binary cross-entropy, written with two columns instead of one.

In [ ]:
probs = np.array([[0.7, 0.2, 0.1],
                  [0.1, 0.1, 0.8],
                  [0.3, 0.4, 0.3]])
y = np.array([0, 2, 1])
rows = np.arange(len(y))
print("probability of the true class:      ", probs[rows, y])
print("the same through one-hot labels:     ", (np.eye(3)[y] * probs).sum(axis=1))

### ✍️ Your turn: `cross_entropy`

Write `cross_entropy(probs, y)`:

- **inputs:** `probs` of shape `(batch, K)`, where each row is a probability distribution, and `y` of shape `(batch,)`, holding integer class indices from `0` to `K - 1`;
- **output:** the mean of $-\log P_{i, y_i}$ as a Python `float`, where each picked probability is first clipped to be at least $10^{-12}$.

Hints: `probs[np.arange(len(y)), y]` picks one entry per row, and `np.clip(values, 1e-12, None)` sets a lower bound only.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below.

In [ ]:
def cross_entropy_reference(probs, y):
    picked = probs[np.arange(len(y)), y]
    return float(-np.mean(np.log(np.clip(picked, 1e-12, None))))

In [ ]:
def cross_entropy(probs, y):
    """Mean of -log(probability of the true class). probs: (batch, K), y: (batch,) ints. Returns a float."""
    # ✍️ Your code here (two lines: pick, then clip, log and average)
    raise NotImplementedError

In [ ]:
cross_entropy = practice("cross_entropy", cross_entropy, cross_entropy_reference, cases=[
    (np.array([[0.7, 0.2, 0.1], [0.1, 0.1, 0.8], [0.3, 0.4, 0.3]]), np.array([0, 2, 1])),
    (np.full((4, 3), 1 / 3), np.array([0, 1, 2, 0])),          # a uniform guess
    (np.array([[0.0, 1.0], [0.5, 0.5]]), np.array([0, 1])),     # edge case: zero probability for the truth
])

**Expected behaviour.** The three rows above cost $-\log 0.7$, $-\log 0.8$ and $-\log 0.4$, a mean of about $0.4987$. A uniform guess over $K$ classes costs exactly $\log K$, which is $\log 3 \approx 1.0986$ for three classes. Remember that number: it is roughly where an untrained network starts. A zero probability for the truth is clipped, so it costs $27.63$ instead of `inf`.

## 6 · When the numbers break: overflow, underflow and log-sum-exp

A `float64` cannot hold numbers above about $1.8 \times 10^{308}$, so $e^{z}$ **overflows** to `inf` once $z > 709.78$. At the other end, probabilities below about $10^{-308}$ (or $5 \times 10^{-324}$ at the very edge) **underflow** to exactly $0$. Large logits trigger both problems.

**Failure 1: a naive softmax overflows.** Computing `np.exp(Z) / np.exp(Z).sum()` with a logit of 1000 gives `inf / inf = nan`. The `softmax` from the recap avoids this by subtracting each row's maximum first. That shift is allowed because $\frac{e^{z_k - c}}{\sum_j e^{z_j - c}} = \frac{e^{z_k}}{\sum_j e^{z_j}}$ for any constant $c$, and after it the largest exponent is $e^0 = 1$.

In [ ]:
Z_big = np.array([[1000.0, 10.0, -5.0]])
with np.errstate(over="ignore", invalid="ignore"):   # silence the overflow warning for this demonstration
    naive = np.exp(Z_big) / np.exp(Z_big).sum(axis=1, keepdims=True)
print("naive softmax:  ", naive)
print("shifted softmax:", softmax(Z_big))

**Failure 2: the probability underflows before you take its logarithm.** Even a stable softmax cannot represent a probability of $e^{-800}$: it rounds to exactly $0$. The clip in `cross_entropy` then reports $27.63$, although the true loss is about $800$. The network would be told it is "a bit wrong" when it is catastrophically wrong.

In [ ]:
Z_wrong = np.array([[0.0, 800.0, -5.0]])   # the true class is 0, but class 1 scores 800 higher
y_wrong = np.array([0])
p_wrong = softmax(Z_wrong)
print("probability of the true class:", p_wrong[0, 0])
print("cross_entropy after the clip: ", cross_entropy(p_wrong, y_wrong))

**The fix: never form the probability.** Write the loss for one example directly in terms of its logits $z_{i1}, \dots, z_{iK}$:

$$-\log P_{i,y_i} = -\log \frac{e^{z_{i,y_i}}}{\sum_j e^{z_{ij}}} = \log \sum_j e^{z_{ij}} \;-\; z_{i,y_i} .$$

The first term, the **log-sum-exp** $\operatorname{LSE}(z_i) = \log \sum_j e^{z_{ij}}$, could still overflow, but subtracting the row maximum $m_i = \max_j z_{ij}$ fixes it exactly:

$$\log \sum_j e^{z_{ij}} = m_i + \log \sum_j e^{z_{ij} - m_i} .$$

Now every exponent is at most $0$ and the largest term is $e^0 = 1$, so the sum lies between $1$ and $K$: it cannot overflow, and its logarithm can never be $\log 0$. No probability is formed, so nothing underflows and no clip is needed. Putting it together,

$$\text{CE} = \frac{1}{n} \sum_{i=1}^{n} \Big( m_i + \log \sum_j e^{z_{ij} - m_i} \;-\; z_{i,y_i} \Big).$$

**Shapes.** `Z` has shape `(batch, K)`. Keep the row maximum as `(batch, 1)` with `keepdims=True` so that `Z - m` subtracts it row by row. The result is a single number.

In [ ]:
z = np.array([1000.0, 999.0, 990.0])
m = z.max()
with np.errstate(over="ignore"):
    print("naive log(sum(exp(z))):        ", np.log(np.sum(np.exp(z))))
print("shifted m + log(sum(exp(z - m))):", m + np.log(np.sum(np.exp(z - m))))

### ✍️ Your turn: `cross_entropy_from_logits`

Write `cross_entropy_from_logits(Z, y)`:

- **inputs:** `Z`, the output layer's logits (before softmax) with shape `(batch, K)`, and `y`, integer class indices with shape `(batch,)`;
- **output:** the mean cross-entropy as a Python `float`, computed with the log-sum-exp formula above and no clipping.

Hints: `m = Z.max(axis=1, keepdims=True)`; the log-sum-exp of every row is `m[:, 0] + np.log(np.exp(Z - m).sum(axis=1))`, with shape `(batch,)`; the true-class logits are `Z[np.arange(len(y)), y]`.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below.

In [ ]:
def cross_entropy_from_logits_reference(Z, y):
    row_max = Z.max(axis=1, keepdims=True)
    log_sum_exp = row_max[:, 0] + np.log(np.exp(Z - row_max).sum(axis=1))
    return float(np.mean(log_sum_exp - Z[np.arange(len(y)), y]))

In [ ]:
def cross_entropy_from_logits(Z, y):
    """Mean cross-entropy from logits Z (batch, K) and integer labels y (batch,), using log-sum-exp."""
    # ✍️ Your code here (about three lines)
    raise NotImplementedError

In [ ]:
cross_entropy_from_logits = practice("cross_entropy_from_logits", cross_entropy_from_logits, cross_entropy_from_logits_reference, cases=[
    (np.array([[2.0, 1.0, 0.1], [0.5, 2.5, -1.0]]), np.array([0, 2])),
    (np.array([[0.0, 800.0, -5.0]]), np.array([0])),            # the underflow case from above
    (np.full((3, 4), 1e4), np.array([0, 3, 1])),                 # edge case: huge but equal logits
])

**Expected behaviour.** For ordinary logits the result agrees with `cross_entropy(softmax(Z), y)` to about 15 decimal places. For the underflow case it returns $800.0$ instead of the clipped $27.63$. Huge but equal logits give exactly $\log 4 \approx 1.3863$: a uniform guess, as they should.

### 🧪 Try it: make the logits more and more extreme

The cell computes the same loss three ways. Raise `scale` to make the network more confident, and watch which method breaks first.

In [ ]:
scale = 1000.0     # try 1.0, 100.0 and 1e6, then run the cell again
Z_demo = scale * np.array([[2.0, -1.0, 0.5], [0.1, 0.3, -2.0]])
y_demo = np.array([1, 0])
with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
    naive_probs = np.exp(Z_demo) / np.exp(Z_demo).sum(axis=1, keepdims=True)
    naive_loss = float(-np.mean(np.log(naive_probs[np.arange(2), y_demo])))
print(f"naive softmax, then log:   {naive_loss:.4f}")
print(f"stable softmax, then clip: {cross_entropy(softmax(Z_demo), y_demo):.4f}")
print(f"from logits (log-sum-exp): {cross_entropy_from_logits(Z_demo, y_demo):.4f}")

At `scale = 1.0` all three agree. At `100.0` they still agree, at about $160$. At `1000.0` the naive version returns `nan`, the clipped version is stuck at $27.63$, and only the log-sum-exp version reports the true loss of $1600$. Libraries such as PyTorch compute cross-entropy from logits for exactly this reason.

## 7 · Choosing a loss

| Task | What the network outputs | Loss | Function in this course |
|---|---|---|---|
| Regression: predict a number | raw values, with no activation on the output | mean squared error | `mse(pred, target)` |
| Binary classification: yes or no | one probability from a sigmoid | binary cross-entropy | `binary_cross_entropy(p, y)` |
| Multiclass: exactly one of $K$ classes | $K$ probabilities from a softmax | cross-entropy | `cross_entropy(probs, y)`, or `cross_entropy_from_logits(Z, y)` |

Some rules of thumb:

- **Match the loss to the output layer.** Cross-entropy expects probabilities (or logits, for the `_from_logits` version). MSE expects raw numbers on the same scale as the target.
- **Avoid MSE for classification.** A confidently wrong prediction costs at most about 1 under MSE, as the table in section 4 showed. Worse, after a sigmoid its gradient *shrinks* exactly when the model is most wrong, as the next cell shows.
- **Compute from logits when you can.** It is the same loss, but it stays exact when the logits are large.
- **Several independent yes-or-no labels per example** (for example, tags on a photo) need one sigmoid and one binary cross-entropy per label, not a softmax.

The next cell previews notebook 06. For one example with true label 1 and logit $z$, the slope of each loss with respect to $z$ is $p - 1$ for binary cross-entropy and $2(p - 1)\,p\,(1 - p)$ for squared error, where $p = \sigma(z)$. Watch what happens as the prediction becomes confidently wrong.

In [ ]:
print(f"{'logit z':>8} {'p = sigmoid(z)':>15} {'BCE slope':>10} {'MSE slope':>10}")
for logit in [2.0, 0.0, -2.0, -5.0, -10.0]:
    p_yes = float(sigmoid(logit))
    print(f"{logit:>8} {p_yes:>15.5f} {p_yes - 1:>10.4f} {2 * (p_yes - 1) * p_yes * (1 - p_yes):>10.4f}")

At $z = -10$ the model is almost certain of the wrong answer. Binary cross-entropy still pulls with a slope close to $-1$, while the squared error's slope has collapsed to about $-0.0001$: the model would barely move. This is the practical reason why classifiers use cross-entropy.

## 8 · Put it together: the loss of an untrained network

Now measure a real network. A freshly initialised 2-64-3 network classifies 300 spiral points from three classes. It has learned nothing yet, so its predictions should be close to a uniform guess and its loss should be close to $\log 3 \approx 1.0986$. Because the logits are moderate, both versions of cross-entropy must agree. Finally, a perfect model and a confidently wrong model mark the two ends of the scale.

In [ ]:
put_rng = np.random.default_rng(42)
X_spiral, y_spiral = make_spirals(100, 3, 0.2, put_rng)
network = init_network([2, 64, 3], put_rng)
probs, cache = forward(network, X_spiral)
Z_out = cache[-1][1]                        # the output layer's logits, shape (300, 3)

loss_from_probs = cross_entropy(probs, y_spiral)
loss_from_logits = cross_entropy_from_logits(Z_out, y_spiral)
print(f"untrained loss: {loss_from_probs:.4f} from probabilities, {loss_from_logits:.4f} from logits (log 3 = {np.log(3):.4f})")
print(f"untrained accuracy: {accuracy(network, X_spiral, y_spiral):.3f} (a random guess scores about 0.333)")

perfect = np.eye(3)[y_spiral]                      # all probability on the true class
confidently_wrong = np.eye(3)[(y_spiral + 1) % 3]  # all probability on a wrong class
print(f"perfect predictions:     {abs(cross_entropy(perfect, y_spiral)):.4f}")
print(f"confidently wrong ones:  {cross_entropy(confidently_wrong, y_spiral):.4f}")
print(f"logits scaled by 1000:   {cross_entropy_from_logits(1000 * Z_out, y_spiral):.1f} from logits, "
      f"{cross_entropy(softmax(1000 * Z_out), y_spiral):.2f} after the clip")

assert abs(loss_from_probs - loss_from_logits) < 1e-9, "both versions of cross-entropy should agree on moderate logits"
assert abs(loss_from_probs - np.log(3)) < 0.25, "an untrained network should score close to a uniform guess"
assert abs(cross_entropy(perfect, y_spiral)) < 1e-9, "perfect predictions should cost nothing"
assert cross_entropy(confidently_wrong, y_spiral) > 27, "confident mistakes should hit the clip at about 27.63"
assert cross_entropy_from_logits(1000 * Z_out, y_spiral) > 100, "huge logits must not be capped by a clip"
print("✅ All checks passed.")

**What to notice.** The untrained network scores close to $\log 3$, so its predictions are close to uniform. Its accuracy is near chance. This gives you a sanity check for any new classifier: **before training, the loss should be close to $\log K$**. A starting loss far above that usually means the initial weights are too large; a loss far below it before any training usually means a bug, such as labels leaking into the inputs.

## Exercises

These go beyond the type-alongs. Each has a check cell, and the reference solutions are in collapsed cells as before.

### Exercise 1 · Mean absolute error

Write `mean_absolute_error(pred, target)`, the mean of $|\hat{y}_i - y_i|$ over every entry, as a `float`. Then answer the open question below the check: on the data `target = [1, 2, 3, 4, 100]` with predictions `[1, 2, 3, 4, 5]`, which loss is dominated by the single outlier, and by how much?

In [ ]:
def mean_absolute_error_reference(pred, target):
    return float(np.mean(np.abs(pred - target)))

In [ ]:
def mean_absolute_error(pred, target):
    """Mean of |pred - target| over every entry. Returns a float."""
    # ✍️ Your code here
    raise NotImplementedError

In [ ]:
mean_absolute_error = practice("mean_absolute_error", mean_absolute_error, mean_absolute_error_reference, cases=[
    (np.array([2.5, 0.0, 2.0]), np.array([3.0, -0.5, 2.0])),
    (np.array([[1.0, -2.0], [0.0, 4.0]]), np.array([[0.0, 2.0], [0.0, 4.0]])),
])
outlier_target = np.array([1.0, 2.0, 3.0, 4.0, 100.0])
outlier_pred = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
print(f"MSE: {mse(outlier_pred, outlier_target):.1f}   MAE: {mean_absolute_error(outlier_pred, outlier_target):.1f}")

### Exercise 2 · Binary cross-entropy from a logit

A sigmoid followed by `binary_cross_entropy` has the same overflow and underflow problems as softmax. Show on paper that, for one example with logit $z$ and label $y$,

$$-\big[y \log \sigma(z) + (1 - y) \log(1 - \sigma(z))\big] = \log(1 + e^{z}) - y z ,$$

then write `binary_cross_entropy_from_logits(z, y)` for arrays `z` and `y` of shape `(batch,)`, returning the mean as a `float`. Hint: `np.logaddexp(0, z)` computes $\log(e^0 + e^z) = \log(1 + e^z)$ without overflow.

In [ ]:
def binary_cross_entropy_from_logits_reference(z, y):
    return float(np.mean(np.logaddexp(0.0, z) - y * z))

In [ ]:
def binary_cross_entropy_from_logits(z, y):
    """Mean binary cross-entropy from logits z (batch,) and 0/1 labels y (batch,), without overflow."""
    # ✍️ Your code here
    raise NotImplementedError

In [ ]:
binary_cross_entropy_from_logits = practice("binary_cross_entropy_from_logits", binary_cross_entropy_from_logits,
                                            binary_cross_entropy_from_logits_reference, cases=[
    (np.array([2.0, -1.0, 0.0]), np.array([1, 0, 1])),
    (np.array([-800.0, 800.0]), np.array([1, 0])),   # confidently wrong in both directions: about 800
])

### Exercise 3 · A stable log-softmax

Write `log_softmax(Z)`, which returns $\log P$ for every entry, shape `(batch, K)`, without ever forming $P$. Each row is $z_{ij} - \operatorname{LSE}(z_i)$. Once it passes, convince yourself that `-np.mean(log_softmax(Z)[np.arange(len(y)), y])` is the same loss as `cross_entropy_from_logits(Z, y)`.

In [ ]:
def log_softmax_reference(Z):
    row_max = Z.max(axis=1, keepdims=True)
    return Z - row_max - np.log(np.exp(Z - row_max).sum(axis=1, keepdims=True))

In [ ]:
def log_softmax(Z):
    """Row-wise log of softmax(Z), computed with the log-sum-exp trick. Z: (batch, K) -> (batch, K)."""
    # ✍️ Your code here
    raise NotImplementedError

In [ ]:
log_softmax = practice("log_softmax", log_softmax, log_softmax_reference, cases=[
    (np.array([[2.0, 1.0, 0.1], [0.5, 2.5, -1.0]]),),
    (np.array([[0.0, 800.0, -5.0]]),),     # entries far below log(1e-308) must stay finite
])

### Exercise 4 · Open questions

1. A network for 10 classes starts training with a loss of $2.30$. Is that what you would expect, and why? What would you suspect if it started at $35$?
2. Our losses average over the batch. What would change in training if they summed instead, and the batch size were doubled?
3. Cross-entropy is never exactly $0$ for a softmax network with finite logits. Why not, and does it matter?

## Common errors and debugging

| Symptom | Likely cause | Fix |
|---|---|---|
| `IndexError: arrays used as indices must be of integer ... type` | the labels `y` are floats | convert them with `y.astype(int)` |
| the loss is a matrix, or has the wrong value | one-hot labels of shape `(batch, K)` passed where integer labels are expected | convert them with `Y.argmax(axis=1)` |
| the loss is `inf` or `nan` | a probability of exactly 0 or 1 reached `np.log`, or a naive softmax overflowed | clip the probabilities, or compute from logits |
| the loss sits at exactly $27.63$ | the clip is active: a probability underflowed to 0 | use `cross_entropy_from_logits` |
| the loss is far too large or far too small for MSE | `pred` has shape `(batch, 1)` and `target` has shape `(batch,)` | reshape so both shapes are identical |
| the loss looks sensible but training goes nowhere | the arguments are swapped, as in `cross_entropy(y, probs)` | keep predictions first and labels second |

The broadcasting bug in the fifth row is the sneakiest, because nothing crashes. The next cell shows it.

In [ ]:
pred_column = np.array([[1.0], [2.0], [3.0]])   # shape (3, 1): a network with one output unit
target_flat = np.array([1.0, 2.0, 3.0])         # shape (3,)
print("shape of pred_column - target_flat:", (pred_column - target_flat).shape)
print("MSE with the shape bug:", round(mse(pred_column, target_flat), 4))
print("MSE after reshaping:   ", mse(pred_column.reshape(-1), target_flat))

The predictions are perfect, yet the buggy MSE is $1.3333$: NumPy compared every prediction with every target. When a loss looks wrong, print the shapes of its inputs before anything else.

## Summary

- A **loss function** turns a batch of predictions into one number to minimise; we always report the mean over the batch.
- **Mean squared error** suits regression. It grows with the square of the error.
- **Binary cross-entropy** and **multiclass cross-entropy** charge $-\log q$ for the probability $q$ given to the truth. Confident mistakes are punished heavily, which keeps a wrong classifier learning.
- $e^z$ **overflows** above $z \approx 709.78$ and tiny probabilities **underflow** to $0$. Subtracting the row maximum makes softmax safe, and the **log-sum-exp** form of cross-entropy avoids forming probabilities at all.
- An untrained $K$-class network should start with a loss close to $\log K$.

You built `mse`, `binary_cross_entropy`, `cross_entropy` and `cross_entropy_from_logits`. The rest of the course uses them under these names.

## Next

A loss tells you *how wrong* the network is, but not *which way to change the weights*. In **06 · Gradients and derivatives** (`06_gradients_and_derivatives.ipynb`) you measure how the loss responds when you nudge a single weight, build a numerical gradient and learn to check gradients like a professional.